In [ ]:
# placeholder
# sources to pull 
https://www.weather.gov.sg/climate-historical-daily/
https://data.gov.sg/datasets?topics=environment&resultId=1459&page=1
https://www.met.gov.my/en/pencerapan/radar-malaysia/

https://www.weather.gov.sg/files/dailydata/DAILYDATA_S24_202508.csv

In [37]:
import requests
import pandas as pd
from io import StringIO
import numpy as np
from datetime import datetime, timezone


In [58]:
def record_scrapper(
        station_id
        ,date
    ):
    batch = str(station_id)+'_'+str(date)
    base_url = 'https://www.weather.gov.sg/files/dailydata/DAILYDATA_'
    url = base_url +batch+'.csv'
    # print(url)

    # url = 'https://www.weather.gov.sg/files/dailydata/DAILYDATA_S64_202508.csv'

    header = { 
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1.15",
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9"
    }

    res = requests.get(url,headers = header)



    print(res.status_code)
    # print(res.headers['Content-Type'])

    df = pd.read_csv(StringIO(res.text))

    return df



In [50]:
def df_cleaner(
        input_df
        ,station_id
    ) :

    # take windspeed , rainfall_mm, timestamp, station id
    input_df['timestamp'] = (input_df['Year'].astype(str))+'-'+input_df['Month'].astype(str).str.zfill(2) + '-'+input_df['Day'].astype(str).str.zfill(2)
    input_df= input_df.drop(columns=['Year','Month','Day'])

    # renaming all columns to be proper column names 
    input_df = input_df.rename(columns={
        'ï»¿Station':'station_name'
        ,'Daily Rainfall Total (mm)' : 'daily_total_rainfall'
        ,'Highest 30 min Rainfall (mm)' : 'highest_30min_rainfall'
        ,'Highest 60 min Rainfall (mm)' : 'highest_60min_rainfall'
        ,'Highest 120 min Rainfall (mm)' : 'highest_120min_rainfall'
        ,'Mean Temperature (Â°C)' : 'mean_temperature'
        ,'Maximum Temperature (Â°C)': 'max_temperature'
        ,'Minimum Temperature (Â°C)' :'min_temperature'
        ,'Mean Wind Speed (km/h)' : 'mean_wind_speed'
        ,'Max Wind Speed (km/h)' : 'max_wind_speed'
        })

    # adding station_id column for downstream table joining
    input_df['station_name'] = input_df['station_name'].astype('string').str.strip().str.lower() # lowercase for standardisation
    input_df['station_name'] = input_df['station_name'].astype('object')
    input_df['station_id'] = station_id # to replace dynamic 

    # rearrrange columns 
    input_df= input_df[['timestamp','station_name','station_id','daily_total_rainfall','highest_30min_rainfall','highest_60min_rainfall','highest_120min_rainfall','mean_temperature','max_temperature','min_temperature','mean_wind_speed','max_wind_speed']]

    # replace - with nan to assert column type if not insert into database will throw error
    input_df = input_df.replace('-',np.nan)



    return input_df

In [ ]:
# api updates 10th of every month 
# date_list= ['202503']
date_list= ['202503,202504,202505,202506,202507,202508']

# for now i pulled the full aws active stations , aws = automatic weather station
station_id_list = [
                    'S104' # admiralty
                    ,'S109' # AMK
                    ,'S24' # changi
                    ,'S121' # CCK south
                    ,'S50' #clementi
                    ,'S107' #ECP
                    ,'S44' # jurong west
                    ,'S117' # jurong island 
                    ,'S108' # marina barrage
                    ,'S111' # newton
                    ,'S116' # pasir panjang
                    ,'S06' # paya lebar
                    ,'S106' # pulau ubin
                    ,'S25' # seletar
                    ,'S102' # semakau island
                    ,'S80' # sembawang
                    ,'S60' # sentosa island
                    ,'S43' # tai seng
                    ,'S23' # tengah
                    ,'S115' # tuas south                                                         
                    ]

def backfill(
    date_list
    ,station_id_list
) : 
    
    for date in date_list :
        for station in station_id_list : 

            # pull api 
            result_df = record_scrapper(station,date)

            # clean df
            clean_df = df_cleaner(result_df,station)


            #################################################################################################
            ## insert database handling here
            ## input variable : clean_df , input datatype : pandas dataframe, can use psycopg2 handler to insert into database
            ## dataframe schema: 

            # timestamp                   object
            # station_name                object
            # station_id                  object
            # daily_total_rainfall       float64
            # highest_30min_rainfall     float64
            # highest_60min_rainfall     float64
            # highest_120min_rainfall    float64
            # mean_temperature           float64
            # max_temperature            float64
            # min_temperature            float64
            # mean_wind_speed            float64
            # max_wind_speed             float64


            ##################################################################################################

            print(f'uploaded station id: {station} for {date}')

In [61]:
backfill(date_list,station_id_list)

200
uploaded station id: S104 for 202503
200
uploaded station id: S109 for 202503
200
uploaded station id: S24 for 202503
200
uploaded station id: S121 for 202503
200
uploaded station id: S50 for 202503
200
uploaded station id: S107 for 202503
200
uploaded station id: S44 for 202503
200
uploaded station id: S117 for 202503
200
uploaded station id: S108 for 202503
200
uploaded station id: S111 for 202503
200
uploaded station id: S116 for 202503
200
uploaded station id: S06 for 202503
200
uploaded station id: S106 for 202503
200
uploaded station id: S25 for 202503
404
uploaded station id: S102 for 202503
200
uploaded station id: S80 for 202503
200
uploaded station id: S60 for 202503
200
uploaded station id: S43 for 202503
200
uploaded station id: S23 for 202503
200
uploaded station id: S115 for 202503
